### Computing the GCD of N numbers and finding a small linear combination

Let's see how the LLL algorithm helps us to find it.

In [4]:
# LLL Algorithm Implementation following An_Introduction_to_Mathematical_Cryptography by Silverman, Figure 7.8

import numpy as np

# The Gram-Schmidt process is used to orthogonalize the basis vectors, which is essential for the LLL algorithm to check the Lovász condition and perform size reduction effectively.
# It must not return the orthonormal basis, but rather the orthogonalized basis, as the LLL algorithm relies on the lengths of these vectors to determine when to swap and reduce.
def gram_schmidt(B):
    B_star = np.zeros_like(B, dtype=float)
    B_star[0] = B[0]
    for i in range(1, len(B)):
        B_star[i] = B[i]
        for j in range(i):
            mu = np.dot(B[i], B_star[j]) / np.dot(B_star[j], B_star[j])
            B_star[i] -= mu * B_star[j]
    return B_star


# The LLL algorithm takes a basis B of a lattice and a parameter delta (which controls the "reduction" level, typically set to 0.75) and returns a reduced basis of the same lattice.
def LLL(B, delta=0.75):
    B_star = gram_schmidt(B)
    k = 1
    n = len(B)
    while k < n:
        for j in range(k-1, -1, -1):
            mu_kj = np.dot(B[k], B_star[j]) / np.dot(B_star[j], B_star[j]) # Projection of B[k] onto B_star[j], normalized by the length of B_star[j], as indicated in the foot note.
            B[k] -= round(mu_kj) * B[j] # Size Reduction
                
        # At this poing, we have reduced B[k] with respect to all previous vectors. Now we check the Lovász condition to decide if we need to swap B[k] and B[k-1].
        if np.dot(B_star[k], B_star[k]) >= (delta - ((np.dot(B[k], B_star[k-1]))/np.dot(B_star[k-1], B_star[k-1]))**2 ) * np.dot(B_star[k-1], B_star[k-1]):
            k += 1
        else:
            B[[k, k-1]] = B[[k-1, k]] # Swap B[k] and B[k-1]
            B_star = gram_schmidt(B)  
            k = max(k-1, 1)
    return B

The idea is that a vector in the lattice generated by the basis $\{1, 0, ..., n_1\}, \{0, 1, ..., n_2\}, ..., \{0, 0, ..., 1, n_p\}$ has the form:

$\{a, b, ..., a*n_1 + ... + p*n_p\}$ where $a, ... , p \in \mathbb{Z}$

So if we find a small vector in this lattice it should be a candidate to be the gcd, the last element of the rows, and also find a small linear combination. Since LLL is not an exact algorithm it could fail with many big numbers for example.

In [5]:
def gcd(numbers):
    
    basis = np.zeros((len(numbers), len(numbers)+1))
    for i in range(0, len(numbers)):
        row = np.zeros(len(numbers)+1)
        row[i] = 1
        row[len(numbers)] = numbers[i]
        basis[i] = row
    
    reduced_basis = LLL(basis)
    #print(reduced_basis)
    
    for i in range(len(numbers)):
        if reduced_basis[i][len(numbers)] == 0:
            continue
        
        sum = 0
        for j in range(len(numbers)+1):
            if j != len(numbers):
               sum += reduced_basis[i][j] * numbers[j]
            else:
                gcd = reduced_basis[i][j]
                if sum ==  gcd:
                    return reduced_basis[i] if gcd > 0 else -1 * reduced_basis[i]
    return [0]*(len(numbers)+1)

We reduce the basis and then we check the last element of the rows to see if it could be a candidate (different than 0), by evaluating the linear combination and seeing it it's equal to the last element of the row. It could happen that it has a negative sign, so we return it positive.

In [6]:
arr = [48, 18, 30, 12, 120]
lincomb = gcd(arr)
    
for i in range(len(lincomb)):
    if i != len(arr):
        print(f"{arr[i]} * ({lincomb[i]}) + ", end="")
    else:
        print(f"= {lincomb[i]}")

48 * (-0.0) + 18 * (1.0) + 30 * (-0.0) + 12 * (-1.0) + 120 * (-0.0) + = 6.0
